# 移動表現を分類する
3モデルの多数決で学習し、初めて見る作品で分類器の予測と読者の判断を比べます。

## データを開く
次の準備セルを1回実行してください。

In [ ]:
#@title データを開く / Load the course dataset
import json
from urllib.request import urlretrieve
from zipfile import ZipFile

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

url = "https://nlp.lang.osaka-u.ac.jp/dh/b/materials/class7-consensus-v1.zip"
archive, _ = urlretrieve(url)
with ZipFile(archive) as bundle:
    passages = [json.loads(line) for line in bundle.open("passages.jsonl")]
    ballots = [json.loads(line) for line in bundle.open("llm-labels.jsonl")]
    reader = json.load(bundle.open("reader-labels.json"))
    with np.load(bundle.open("embeddings.npz")) as data:
        vectors = dict(zip(data["ids"], data["vectors"].astype("float32")))
    rubric = bundle.read("rubric-ja.txt").decode()
items = {p["id"]: p for p in passages}
train = [b for b in ballots if items[b["id"]]["split"] == "train" and b["label"]]
validation = [b for b in ballots if items[b["id"]]["split"] == "dev" and b["label"]]
valid = [b for b in validation if items[b["id"]].get("sampling") != "negation-cue"]
targeted = [b for b in validation if items[b["id"]].get("sampling") == "negation-cue"]
train_x = [vectors[b["id"]] for b in train]
train_y = [b["label"] for b in train]
valid_x = [vectors[b["id"]] for b in valid]
valid_y = [b["label"] for b in valid]
print("Training:", len(train), "Original validation:", len(valid), "Negation validation:", len(targeted), "Reader panel:", len(reader))

## 1. 判断基準と照らして読む
まず、判断基準と意見が分かれた学習例を読み、自分のラベルを決めてください。

In [ ]:
print(rubric)
case = next(b for b in train if b["agreement"] == 2)
p = items[case["id"]]
print(p["previous"])
print(p["left"] + "【" + p["target"] + "】" + p["right"])
print(p["next"])

In [ ]:
for name, vote in case["votes"].items():
    if vote:
        print(name, vote["label"], vote["evidence"])
print("Consensus:", case["label"])

## 2. 検証用の用例で設定を選ぶ
対象語と文脈を表す埋め込みを使います。まず標準の重みで実行し、次に `None` を `"balanced"` に変えて再実行してください。どのラベルで再現率が変わりましたか。評価用を開く前に、採用する設定と理由を記録してください。

In [ ]:
weights = None
model = LogisticRegression(class_weight=weights, max_iter=2000)
model.fit(train_x, train_y)
predictions = model.predict(valid_x)
present = [label for label in ["CT+", "CT-", "Uu", "NM"] if label in valid_y]
print(classification_report(valid_y, predictions, labels=present, zero_division=np.nan))

### 否定表現を手掛かりに追加した用例でCT−を調べる
元の検証用にはCT−がありません。否定表現を手掛かりに集めた別の検証用のうち、CT−を何件拾えましたか。

In [ ]:
negatives = [b for b in targeted if b["label"] == "CT-"]
negative_predictions = model.predict([vectors[b["id"]] for b in negatives])
print("CT- recovered:", sum(negative_predictions == "CT-"), "/", len(negatives))

## 3. 読者のラベルと比べる
作品単位で学習用・検証用から除いた20箇所です。無作為に選んだ12箇所のうち1箇所は対象語の範囲に問題があり、集計から除きます。残り8箇所は旧Qwenが少数ラベルを付けた例から選びました。二つの集計を比べてください。

In [ ]:
for cohort in ["random", "rare-9b"]:
    ids = [i for i in reader if reader[i]["cohort"] == cohort and reader[i]["label"] != "defer"]
    gold = [reader[i]["label"] for i in ids]
    predicted = model.predict([vectors[i] for i in ids])
    print(cohort, sum(predicted == gold), "/", len(ids))

読者のラベルと予測が異なる例では、対象語と前後の文を読み、どの表現がどちらの判断を支えるか説明してください。

In [ ]:
for i, reading in reader.items():
    predicted = model.predict([vectors[i]])[0]
    if reading["label"] != "defer" and reading["label"] != predicted:
        p = items[i]
        print(p["title"], p["previous"], p["left"] + "【" + p["target"] + "】" + p["right"], p["next"], sep="\n")
        print("Reader:", reading["label"], reading["evidence"], "/ Prediction:", predicted)
        break